#### Workshop instructions notebook
This notebook serves as a guide through the workshop materials; follow its cells below to navigate through the steps.

**NOTE**:
Before starting, rename the file, config_template.yml, to config.yml and populate the Unity Catalog values
 - uc_catalog 
 - uc_schema
 - table_postfix  

In [0]:
%pip install openai-agents databricks-openai nest_asyncio "typing_extensions>=4.12" -q
%restart_python

#### 1. Playground
Navigate to the **Playground** icon on the left toolbar. Spend some time reviewing the Playground. Notice that different models can be chosen via the dropdown menue at the top. Choose the model: **GPT-5.5**

<img src="img/playground_icon.png" width="200" />

Walk through the examples at the bottom under **Start with an example**:

<img src="img/start_with_an_example_playground.png" width="800" />


#### 2. AI Gateway. 
Navigate to the AI Gateway icon on the left tool bar.  

<img src="img/ai_gateway_icon.png" width="150" />

Spend some time reviewing the AI Gateway UI, then click the model, **claude-opus-5**. Review the configuration and governance options for the model in AI Gateway.

#### 3. Selecting and calling an foundation model from AI Gateway
Update the DATABRICKS_HOST to be your workspace URL (Example: https://aaa-bbb-111.cloud.databricks.com)

In [0]:
from openai import OpenAI

DATABRICKS_TOKEN = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
DATABRICKS_HOST = "https://<your-workspace>.cloud.databricks.com"
BASE_URL = f"{DATABRICKS_HOST}/ai-gateway"
MODEL = "system.ai.gpt-5-5"

client = OpenAI(
  api_key=DATABRICKS_TOKEN,
  base_url=f"{DATABRICKS_HOST}/ai-gateway/mlflow/v1"
)

response = client.responses.create(
  model=MODEL,
  max_output_tokens=2048,
  input=[
    {
      "role": "user",
      "content": [{"type": "input_text", "text": "Hello!"}]
    },
    {
      "role": "assistant",
      "content": [{"type": "output_text", "text": "Hello! How can I assist you today?"}]
    },
    {
      "role": "user",
      "content": [{"type": "input_text", "text": "What is Databricks?"}]
    }
  ]
)

if response.output_text:
    print(response.output_text)
else:
    print("No output_text found. Full response:")
    print(response)

#### 4. Creating a tool in Unity Catalog for agents to call
View the tool in Unity Catalog

In [0]:
import yaml
import os

# Load workshop config
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

UC_CATALOG = config["uc_catalog"]
UC_SCHEMA = config["uc_schema"]

# Create a simple addition tool in Unity Catalog
spark.sql(f"""
CREATE OR REPLACE FUNCTION {UC_CATALOG}.{UC_SCHEMA}.add_numbers(a DOUBLE, b DOUBLE)
RETURNS DOUBLE
RETURN a + b
""")

print(f"Created function {UC_CATALOG}.{UC_SCHEMA}.add_numbers")

#### 5. Configuring a tool calling agent
Examine the MLFlow trace produced by the model call. See the [MLFlow tracing documentation](https://docs.databricks.com/aws/en/mlflow3/genai/tracing/). This is an important part of agentic development on Databricks. Note that a [getting started demo](https://docs.databricks.com/aws/en/mlflow3/genai/getting-started/tracing/tracing-notebook) is available.

Traces enable:
* **Debugging** — See exactly what your agent did at each step, including LLM inputs/outputs, tool calls, and retrieval results
* **Latency analysis** — Identify which steps in your agent's chain are slow (e.g., a retrieval call vs. an LLM generation)
* **Cost visibility** — Track token usage per request to understand and control spending
* **Quality evaluation** — Review actual agent responses alongside the context they used, making it easy to spot hallucinations or missed information
* **Production monitoring** — Catch regressions early by comparing trace patterns across deployments

In [0]:
import os
import asyncio
import nest_asyncio
from agents import Agent, Runner
from databricks.sdk import WorkspaceClient
from databricks_openai.agents import McpServer

import mlflow
mlflow.openai.autolog()

nest_asyncio.apply()

# The Agents SDK reads credentials from environment variables
os.environ["OPENAI_API_KEY"] = DATABRICKS_TOKEN
os.environ["OPENAI_BASE_URL"] = f"{DATABRICKS_HOST}/ai-gateway/mlflow/v1"

w = WorkspaceClient()

# Use the Databricks managed MCP server for Unity Catalog functions.
# The agent automatically discovers the tool schema from the UC function metadata.
async def main():
    async with McpServer(
        url=f"{DATABRICKS_HOST}/api/2.0/mcp/functions/{UC_CATALOG}/{UC_SCHEMA}/add_numbers",
        name="uc-add-numbers",
        workspace_client=w,
    ) as uc_server:
        agent = Agent(
            name="Calculator",
            instructions="You are a calculator. Use the add_numbers tool to answer math questions.",
            model=MODEL,
            mcp_servers=[uc_server],
        )
        result = await Runner.run(agent, "What is 42 + 58?")
        print(result.final_output)

asyncio.run(main())

#### 6. Creating and configuring a Genie Agent for text-to-sql over Delta tables

<img src="img/genie_architecture_diagram.png?v=2" width="900" />

##### 6a. Generting synthetic data
Paste the below prompt in your Genie Code pane to the right...

**PROMPT**:  
"Run the data notebooks in the data/ folder to build the example Delta tables. Check AGENT.md and config.yml for setup details."

This will instruct Genie to build synthetic datasets and write them to Delta tables in Unity Catalog.

Aftere Genie completes the task, query the data below.

In [0]:
import yaml
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["uc_catalog"]
schema = config["uc_schema"]
prefix = config["table_prefix"]
postfix = config["table_postfix"]

tables = ["customers", "products", "stores", "orders", "order_items", "inventory"]

rows = []
for table in tables:
    fqn = f"{catalog}.{schema}.{prefix}_{table}{postfix}"
    count = spark.table(fqn).count()
    rows.append((fqn, count))

display(spark.createDataFrame(rows, ["table_name", "row_count"]))

In [0]:
table_name = "<catalog.schema.table_name>"
display(spark.table(table_name))

##### 6b. Provisioning the Genie Agent

Select the Genie Agents icon from the left panel.

<img src="img/genie_icon.png" width="150" />

Click **New** and select all the tables you created in step 6a.

<img src="img/genie_attach_tables.png" width="600" />

Then, spend some time reviewing the Genie Agent UI.

###UPDATE the genie_agent_id field in your config.yml

##### 6c. Configuring the Genie Agent
Copy the file **data/genie_config.md** to your Genie Agent's instructions. 

<img src="img/genie_instructions.png" width="600" />

After pasting the instuctions, click **Save**. Then, click **Improve with Genie**. Notice how Genie moves content to different areas of the Genie Agent's configuration to optimize the agent's performance.

##### 6d. Testing the Genie Agent
Spend some time interating with your Genie Agent. The example questions provide a starting point.

##### 6e. Query the Genie Agent remotely
Your Genie Agent can be called remotely, meaning external agents can leverage its text-to-sql capabilities.   

**Note**: You must update the genie_agent_id field in your config.yml for the below code to work.  

<img src="img/genie_agent_as_tool.png?v=2" width="800" />



In [0]:
import yaml
import os
import time
from databricks.sdk import WorkspaceClient

# Load config
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

genie_space_id = config["genie_agent_id"]

w = WorkspaceClient()

# Start a conversation with the Genie Agent
conversation = w.genie.start_conversation(
    space_id=genie_space_id,
    content="What are the top 5 products by total revenue?"
)

conversation_id = conversation.conversation_id
message_id = conversation.message_id
print(f"Conversation started: {conversation_id}")

# Poll for the response
while True:
    message = w.genie.get_message(
        space_id=genie_space_id,
        conversation_id=conversation_id,
        message_id=message_id
    )
    status = message.status.value if hasattr(message.status, 'value') else str(message.status)
    if status in ("COMPLETED", "FAILED"):
        break
    print(f"  Status: {status}...")
    time.sleep(3)

print(f"\nFinal status: {status}")

# Display the results
if message.attachments:
    for attachment in message.attachments:
        if attachment.text:
            print(f"\nGenie response:\n{attachment.text.content}")
        if attachment.query:
            print(f"\nGenerated SQL:\n{attachment.query.query}")

##### 6f. Query the Genie Agent using Databricks managed Genie Agent MCP server  
**What is MCP?**  
[Model Context Protocol (MCP)](https://docs.databricks.com/aws/en/agents/mcp-tools/connect-clients) is an open standard that defines how agents discover and call tools. It uses a server/client model: an **MCP server** exposes one or more tools behind a URL, and an **MCP client** (your agent framework) connects to that URL, discovers the available tools, and calls them on behalf of the agent. Build the tool once, and any MCP-compatible agent can use it — regardless of framework.

**Key concepts:**
* **Tool discovery** — When an agent connects to an MCP server, it automatically learns what tools are available and how to call them. No manual tool schema definitions needed.
* **Transport** — MCP servers communicate over Streamable HTTP (SSE), not plain REST. This means you need an MCP-aware client — you can't just `curl` the endpoint.
* **Security** — Databricks MCP servers enforce Unity Catalog permissions on every request. Agents can only access data their user is authorized to see.

**Databricks managed MCP servers:**  
Databricks provides [managed MCP servers](https://docs.databricks.com/aws/en/agents/mcp-tools/managed-mcp) out of the box so you don't have to build your own. Available servers include:
* [Genie Agent MCP server](https://docs.databricks.com/aws/en/agents/mcp-tools/genie-agent) — wraps any Genie Space as a callable tool (used below)
* Unity Catalog function MCP server — exposes UC functions as tools
* Vector Search MCP server — exposes indexes for retrieval

MCP is most valuable when you're connecting multiple clients or frameworks to the same Genie Agent, or when you want to minimize custom integration code. Compare Cell 20 (direct SDK, \~40 lines of manual polling) with Cell 22 below, where MCP abstracts that away.

The below example uses the [OpenAI Agents SDK](https://openai.github.io/openai-agents-python/), a lightweight agent framework that sits on top of the base OpenAI Python SDK. Where the base SDK (used in cells 6 and 11) gives you direct API calls that you orchestrate yourself, the Agents SDK adds:
* **Automatic tool-calling loops** — the framework calls tools, feeds results back to the model, and repeats until the model is done
* **Built-in MCP support** — point at an MCP server URL and the agent discovers and calls its tools automatically
* **Agent handoffs** — route between specialized agents (e.g., a data agent and a document agent)

It is async-first (`await Runner.run(...)`) and requires `nest_asyncio` in Databricks notebooks since the notebook already has a running event loop.

In [0]:
import os
import asyncio
import nest_asyncio
import yaml
from agents import Agent, Runner
from agents.tracing import set_tracing_disabled
set_tracing_disabled(True)
from databricks.sdk import WorkspaceClient
from databricks_openai.agents import McpServer

nest_asyncio.apply()

# Load config
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

genie_space_id = config["genie_agent_id"]

w = WorkspaceClient()
host = w.config.host

# Configure the OpenAI Agents SDK to route through Databricks AI Gateway
token = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiToken().get()
os.environ["OPENAI_API_KEY"] = token
os.environ["OPENAI_BASE_URL"] = f"{host}/ai-gateway/mlflow/v1"

async def main():
    async with McpServer(
        url=f"{host}/api/2.0/mcp/genie/{genie_space_id}",
        name="genie-agent",
        workspace_client=w,
    ) as genie_server:
        agent = Agent(
            name="Data analyst agent",
            instructions="You are a data analyst. Use the Genie tool to query structured data and answer questions.",
            model="system.ai.claude-sonnet-4-5",
            mcp_servers=[genie_server],
        )
        result = await Runner.run(agent, "What are the top 5 products by total revenue?", max_turns=25)
        print(result.final_output)

asyncio.run(main())

#### 7. Processing documents

<img src="img/document_processing_workflow.png" width="1200" />

##### 7a. Copy documents to a Unity Catalog Volume

Paste the below prompt in your Genie Code pane to the right...

**PROMPT**:   
"Upload the PDF documents to the Unity Catalog volume. Check AGENT.md and config.yml for setup details."

This will instruct Genie to copy the PDF documents to a Unity Catalog Volume.

The below cell lists the files in the Volume

In [0]:
import yaml
import os

notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")

with open(config_path, "r") as f:
    config = yaml.safe_load(f)

catalog = config["uc_catalog"]
schema = config["uc_schema"]
postfix = config["table_postfix"]
volume_name = f"{config['volume_name']}{postfix}"

volume_path = f"/Volumes/{catalog}/{schema}/{volume_name}"
print(f"Volume: {volume_path}\n")

files = dbutils.fs.ls(volume_path)
for f in files:
    if f.name.endswith(".pdf"):
        print(f"  {f.name}  ({f.size:,} bytes)")

print(f"\nTotal PDF files: {sum(1 for f in files if f.name.endswith('.pdf'))}")

##### 7b. Parse the documents into raw text and convert the text into text chunks.
Open the notebook, **process_documents/01_parse_and_chunk**, and run all the cells in sequence.

##### 7c. Convert the text chunks to embeddings, provision a vector search endpoint, and populate a vector search index
Open the notebok, **process_documents/02_create_vector_index**, and run all the cells in sequence.

#### 8. Deploy a chatbot on Databricks apps

<img src="img/agents_on_apps_architecture.png" width="1000" />

#### Before proceeding with the below steps...
Click on the **Playground** tab, choose GPT-5-5, then select your Genie Agent and Vector Index from the tools dropdown. Then, start asking your agent questions. In the following cells, we will deploy a similar agent manually on Databricks Apps.

##### 8a. Create an MLflow Experiment
Select **Agents and LLM apps**. Give your Experiment a name and choose the In the Experiment option.

<img src="img/mlflow_experiments_icon.png" width="150" />

Take some time to look throgh the MLflow Experiment UI

##### 8b. Deploy an Agent on Databricks Apps

<img src="img/agent_icon.png" width="150" />

Select **Create Agent** and **Code your own agent**

Select the MlFlow Experiment previously created. Your app will be deployed and you will see the below UI for your application.

<img src="img/databricks_apps_deployed_ui.png" width="800" />

Click the link under App status to navigate to the Chatbot front end. Click "View source" to navigate to the source code, which you will edit in the following cells. 

###UPDATE the config.yml parameter, app_code_dir, with your app's directory

##### 8c. Review the app's agent code in **agent_server/agent.py**.  
Click on "View source" to navigate to the apps source code, under the folder agent_server, review the file, agent.py. The code should look similar to some prior cells you ran in this notebook.

##### 8d. Update the code in **agent_server/agent.py** to connect to Databricks managed MCP servers for Databricks Agents and AI Search. 

Paste the below prompt in your Genie Code pane to the right...

**PROMPT**:   
"Set up the tool-calling agent with Genie and AI Search MCP connections in my Databricks App. Read config.yml for app_code_dir and resource values. Copy agents_on_apps/agent_mcp_reference.py into the app's agent_server/ directory as agent_tool_calling.py, replacing placeholder values. Configure agent.py as the thin architecture router described in AGENT.md."

This will instruct Genie to update the agent.py file containing the agent code to access the Genie Agent and Vector Search Index previously created.

Re-deploy the app from the apps main UI. Once re-deployed, ask question about the data in the Chat inerface. Note that what we deployed here is a tool calling agent.

<img src="img/tool_calling_agent_architecture.png" width="800" />


**Agent template repository**: View the [Agent templates](https://github.com/databricks/app-templates/tree/main) available through Databricks.

In [0]:
# ---------------------------------------------------------------------------
# Call the deployed agent programmatically using DatabricksOpenAI
#
# Authenticates via OAuth M2M using a service principal (client_id + client_secret).
# The WorkspaceClient exchanges these credentials for an OAuth token, which is
# required to call Databricks Apps endpoints (runtime PAT auth is not accepted).
#
# Store your SP credentials in a Databricks secret scope before running:
#   databricks secrets put-secret <your-scope> client-id     --string-value <SP_CLIENT_ID>
#   databricks secrets put-secret <your-scope> client-secret --string-value <SP_CLIENT_SECRET>
#
# Docs: https://docs.databricks.com/aws/en/agents/custom-agents/query-agent/
# ---------------------------------------------------------------------------

import yaml
import os
import re
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI

client_id = dbutils.secrets.get(scope="<scope-name>", key="client_id")
client_secret = dbutils.secrets.get(scope="<scope-name>", key="client_secret")

# Load config
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

# Derive app name from app_code_dir
# Path pattern: .../databricks_apps/<app_name>_<timestamp>/<template>/
app_code_dir = config["app_code_dir"]
app_dir_segment = app_code_dir.split("/databricks_apps/")[1].split("/")[0]
app_name = re.match(r"(.+?)_\d{4}_\d{2}_\d{2}", app_dir_segment).group(1)
print(f"App: {app_name}\n")

# Authenticate with OAuth M2M — passes SP credentials so the SDK
# fetches an OAuth token instead of using the notebook's runtime PAT.
# The host parameter is required so the SDK uses M2M auth instead of
# falling back to the notebook's runtime PAT (which doesn't support OAuth).
host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
w = WorkspaceClient(
    host=host,
    client_id=client_id,
    client_secret=client_secret,
)
client = DatabricksOpenAI(workspace_client=w)

# Call the deployed agent using the Responses API
# The model name uses the "apps/" prefix to route to the Databricks App
response = client.responses.create(
    model=f"apps/{app_name}",
    input=[{"role": "user", "content": "What are the top 5 products by total revenue?"}],
)

print(response.output_text)

##### 8e. Migrate the app to a supervisor architecture. 
**PROMPT:**

"Add the supervisor agent architecture to my Databricks App. Read config.yml for app_code_dir and resource values. Copy agents_on_apps/agent_supervisor_reference.py into the app's agent_server/ directory as agent_supervisor.py, replacing placeholder values. Also copy agents_on_apps/history_reference.py into agent_server/history.py to fix the multi-turn conversation bug. Follow AGENT.md for the pattern."

Then, adjust the agent.py file for your app to import from the new supervisor architecture script.

<img src="img/supervisor_agent_architecture.png" width="800" />


##### When to use each architecture

| | Tool-Calling Agent | Supervisor Agent |
|---|---|---|
| **How it works** | Single agent sees all tools and picks which to call | Supervisor delegates to specialist sub-agents via handoffs |
| **Best for** | Few tools, no instruction conflicts | Specialists that need different instructions, models, or isolated contexts |
| **Routing** | Model picks tools from a flat list guided by one instruction set | Supervisor matches the question type to the right specialist |
| **Code complexity** | Simplest — one agent, one instruction block | Slightly more — but still a small delta in the OpenAI Agents SDK |
| **Failure isolation** | A bad tool call can pollute the whole context window | Each specialist has its own context; errors stay contained |
| **Observability** | One flat trace | Trace shows which specialist handled the request |

**Rule of thumb**: Start with tool-calling. Move to a supervisor when you find yourself writing routing logic *inside* the instructions ("if the question is about X, use tool Y") or when specialists need conflicting guidance.

In **step 8d** you deployed a tool-calling agent. In **step 8e** you added a supervisor that delegates to a data analyst and a document expert. To switch between them, edit `agent_server/agent.py` — uncomment the architecture you want and redeploy.

#### 9. (Optional) LangGraph supervisor architecture
##### Why LangGraph?

Steps 8d and 8e used the OpenAI Agents SDK — a lightweight, async-first framework where multi-agent routing is a one-liner (`handoffs=[...]`). LangGraph is a lower-level alternative that models your agent as a **state machine** (a directed graph of nodes and edges). This gives you capabilities the Agents SDK intentionally abstracts away:

| | OpenAI Agents SDK | LangGraph |
|---|---|---|
| **Agent definition** | Declarative (`Agent(handoffs=[...])`) | Graph nodes, edges, and a compiled `StateGraph` |
| **Routing** | Model-driven handoffs — the LLM picks the specialist | Explicit edges, conditional functions, or a `create_supervisor` helper |
| **State management** | Automatic (framework-managed context) | You define the state schema (`MessagesState` or custom) and control what flows between nodes |
| **Tool integration** | Native MCP support (`mcp_servers=[...]`) | Wrap tools as `StructuredTool`; call APIs directly in tool functions |
| **Code volume** | ~200 lines for a two-specialist supervisor | ~400 lines for the same — graph construction |
| **Best for** | Fast iteration, standard routing patterns | Custom graph topologies, conditional branching, cycles, shared state across nodes |

**When to reach for LangGraph**: You need conditional edges ("if retrieval confidence < 0.7, re-query with a different strategy"), cycles ("keep refining until quality threshold is met"), custom state that persists across nodes, or fine-grained control over which messages each specialist sees.

The LangGraph supervisor implementation lives in `langgraph_agent/` and is imported directly into this notebook — no app deployment needed. The cells below install the dependencies, display the supervisor's graph, and invoke it with a test question.

In [0]:
%uv pip install databricks-langchain langgraph langgraph-supervisor
%restart_python

In [0]:
import sys
import os
import mlflow
from IPython.display import Image, display

mlflow.autolog(disable=True)
mlflow.tracing.disable()

# Add the workshop directory to the path so langgraph_agent is importable
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
workshop_dir = f"/Workspace{os.path.dirname(notebook_path)}"
if workshop_dir not in sys.path:
    sys.path.insert(0, workshop_dir)

from langgraph_agent.supervisor import supervisor

# Display the supervisor's graph
display(Image(supervisor.get_graph().draw_mermaid_png()))

In [0]:
import mlflow

mlflow.tracing.enable()
mlflow.langchain.autolog()

result = supervisor.invoke({"messages": [{"role": "user", "content": "What is our return policy?"}]})
print(result["messages"][-1].content)

#### 10. Agent evaluation

In [0]:
# ---------------------------------------------------------------------------
# Agent Evaluation with mlflow.genai.evaluate()
#
# Calls the deployed GlowMart agent on a small set of test questions and
# scores each response with built-in LLM judge scorers (relevance, safety,
# custom guidelines) plus a code-based latency scorer.
# ---------------------------------------------------------------------------

import mlflow
import yaml
import os
import re
from databricks.sdk import WorkspaceClient
from databricks_openai import DatabricksOpenAI
from mlflow.genai.scorers import scorer, Guidelines
from mlflow.entities import Feedback

# --- Load config and set up DatabricksOpenAI client (same auth as cell 35) ---
notebook_path = dbutils.notebook.entry_point.getDbutils().notebook().getContext().notebookPath().get()
config_path = os.path.join(f"/Workspace{os.path.dirname(notebook_path)}", "config.yml")
with open(config_path, "r") as f:
    config = yaml.safe_load(f)

app_code_dir = config["app_code_dir"]
app_dir_segment = app_code_dir.split("/databricks_apps/")[1].split("/")[0]
app_name = re.match(r"(.+?)_\d{4}_\d{2}_\d{2}", app_dir_segment).group(1)

client_id = dbutils.secrets.get(scope="<secret-scope>", key="client-id")
client_secret = dbutils.secrets.get(scope="<secret-scope>", key="client-secret")

host = dbutils.notebook.entry_point.getDbutils().notebook().getContext().apiUrl().get()
w = WorkspaceClient(host=host, client_id=client_id, client_secret=client_secret)
client = DatabricksOpenAI(workspace_client=w)

# --- Set MLflow experiment (find the user's workshop experiment) ---
experiments = mlflow.search_experiments(max_results=20)
workshop_exps = [e for e in experiments if e.experiment_id != "0" and "Default" not in e.name]
if workshop_exps:
    mlflow.set_experiment(workshop_exps[0].name)
    print(f"Using experiment: {workshop_exps[0].name}")
else:
    print("No workshop experiment found — using default.")

# --- Predict function: calls the deployed agent via Responses API ---
def predict_fn(query: str) -> str:
    response = client.responses.create(
        model=f"apps/{app_name}",
        input=[{"role": "user", "content": query}],
    )
    return response.output_text

# --- Custom code-based scorer: latency under 15 seconds ---
@scorer
def latency_under_15s(inputs, outputs, trace) -> Feedback:
    duration_ms = trace.info.execution_duration
    passed = duration_ms <= 15000
    return Feedback(
        value=passed,
        rationale=f"Latency {duration_ms / 1000:.1f}s {'within' if passed else 'exceeds'} 15s threshold",
    )

# --- Evaluation dataset: 4 GlowMart questions ---
# Two structured-data questions (Genie) and two document questions (AI Search)
eval_data = [
    {
        "inputs": {"query": "What are the top 5 products by total revenue?"},
        "expectations": {"expected_facts": ["revenue", "product"]},
    },
    {
        "inputs": {"query": "What is the return policy?"},
        "expectations": {"expected_facts": ["return", "days"]},
    },
    {
        "inputs": {"query": "What are the loyalty program tiers?"},
        "expectations": {"expected_facts": ["Bronze", "Silver", "Gold", "Platinum"]},
    },
    {
        "inputs": {"query": "Which stores need the most inventory reorders?"},
        "expectations": {"expected_facts": ["store", "inventory"]},
    },
]

# --- Run evaluation ---
print(f"Evaluating {len(eval_data)} questions against app: {app_name}\n")

result = mlflow.genai.evaluate(
    data=eval_data,
    predict_fn=predict_fn,
    scorers=[
        mlflow.genai.scorers.RelevanceToQuery(),
        mlflow.genai.scorers.Safety(),
        Guidelines(
            name="citation",
            guidelines=["The response should mention which tool or data source was used to answer the question."],
        ),
        latency_under_15s,
    ],
)

# --- Display per-question results ---
import json

df = result.tables["eval_results"].copy()
# Stringify the assessments column (nested objects that can't convert to Arrow)
if "assessments" in df.columns:
    df["assessments"] = df["assessments"].apply(
        lambda a: json.dumps(a, default=str) if a is not None else None
    )
display(df)